In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os

from tensorflow.python.ops.numpy_ops import np_config
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense, Input, Dropout, LayerNormalization

I0000 00:00:1788193038.959425     385 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788193039.213169     385 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788193040.960201     385 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import sys
import tensorflow as tf

print(sys.executable)
print(tf.config.list_physical_devices("GPU"))

/home/rkim8/translation-venv/bin/python3
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
eng = pd.read_csv("Data/small_vocab_en.csv", sep="|", header=None)
fr = pd.read_csv("Data/small_vocab_fr.csv", sep="|", header=None)

In [4]:
eng.isnull().sum()

0    0
dtype: int64

In [5]:
fr.isnull().sum()

0    0
dtype: int64

No Null Values

In [6]:
eng.head()

,0
0,"new jersey is sometimes quiet during autumn , ..."
1,the united states is usually chilly during jul...
2,"california is usually quiet during march , and..."
3,the united states is sometimes mild during jun...
4,"your least liked fruit is the grape , but my l..."


In [7]:
eng.iloc[1]

0    the united states is usually chilly during jul...
Name: 1, dtype: str

In [8]:
english_sentences = eng[0].values
french_sentences = fr[0].values

In [9]:
type(english_sentences)

pandas.arrays.StringArray

In [10]:
for i, _ in enumerate(english_sentences):
    english_sentences[i] = "sos " + str(english_sentences[i]) + " eos."
    french_sentences[i] = "sos " + str(french_sentences[i]) + " eos."

#adding start of sentence (sos) and end of sentence (eos) tags

In [11]:
len(english_sentences)

137860

In [12]:
num_words = 1000
tokenizer_en = Tokenizer(num_words=num_words, filters='!#$%&()*+,-/:;<=>@«»""[\\]^_`{|}~\t\n')
tokenizer_en.fit_on_texts(english_sentences)
english_sentences = tokenizer_en.texts_to_sequences(english_sentences)

word_index = tokenizer_en.word_index
print(f"The number of words in the English vocabulary: {len(word_index)}")

The number of words in the English vocabulary: 228


In [13]:
tokenizer_fr = Tokenizer(num_words=num_words, filters='!#$%&()*+,-/:;<=>@«»""[\\]^_`{|}~\t\n')
tokenizer_fr.fit_on_texts(french_sentences)
french_sentences = tokenizer_fr.texts_to_sequences(french_sentences)

word_index_fr = tokenizer_fr.word_index
print(f"The number of words in the French vocabulary: {len(word_index_fr)}")

The number of words in the French vocabulary: 350


In [14]:
print(len(tokenizer_en.word_index))

228


# example on how tokenizer works
tokenizer = Tokenizer(num_words=10)

tokenizer.fit_on_texts(["the cat sat", "the dog ran"])

tokenizer.word_index   # {'the': 1, 'cat': 2, 'sat': 3, 'dog': 4, 'ran': 5, ...}

tokenizer.texts_to_sequences(["the cat ran"])   # [[1, 2, 5]]

In [15]:
print(english_sentences[1])

[2, 8, 23, 24, 1, 12, 65, 7, 46, 10, 6, 1, 12, 54, 5, 48, 4, 3]


In [16]:
print(french_sentences[1])

[2, 7, 35, 34, 1, 15, 22, 5, 52, 9, 6, 98, 72, 5, 54, 4, 3]


In [17]:
lengths = [len(s) for s in english_sentences]
print(max(lengths))

lengths_fr = [len(s) for s in french_sentences]
print(max(lengths_fr))

18
24


In [18]:
english_sentences = pad_sequences(english_sentences, maxlen = max(lengths_fr), padding='post', truncating='post')
french_sentences = pad_sequences(french_sentences, maxlen= max(lengths_fr), padding='post', truncating='post')

#add padding

## Train / Validation / Test Split

Transformer II uses a single **85% train / 10% validation / 5% test** split (fixed `random_state=42`) instead of training on the full dataset like Transformer I. The tokenizer and padding above are unchanged (fit on the full corpus), so the vocabulary and preprocessing are identical to Transformer I — only the split used for training/evaluation differs.

In [19]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

indices = np.arange(len(english_sentences))

train_idx, test_idx = train_test_split(indices, test_size=0.05, random_state=RANDOM_STATE)
train_idx, val_idx = train_test_split(train_idx, test_size=10/95, random_state=RANDOM_STATE)
# 10/95 of the remaining 95% = 10% of the total, leaving 85% for training

english_train, french_train = english_sentences[train_idx], french_sentences[train_idx]
english_val, french_val = english_sentences[val_idx], french_sentences[val_idx]
english_test, french_test = english_sentences[test_idx], french_sentences[test_idx]

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")


Train: 117181  Val: 13786  Test: 6893


In [20]:
embedding_dim = 128

en_vocab_size = len(tokenizer_en.word_index) + 1  # +1 since index 0 is reserved
fr_vocab_size = len(tokenizer_fr.word_index) + 1

en_embedding_layer = Embedding(input_dim=en_vocab_size, output_dim=embedding_dim)
fr_embedding_layer = Embedding(input_dim=fr_vocab_size, output_dim=embedding_dim)

english_embedded = en_embedding_layer(english_sentences)  # (num_sentences, 24) -> (num_sentences, 24, embedding_dim)
french_embedded = fr_embedding_layer(french_sentences)

print(english_embedded.shape)
print(french_embedded.shape)

I0000 00:00:1788193048.734978     385 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


(137860, 24, 128)
(137860, 24, 128)


$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

**This is the standard sinusoidal positional encoding from "Attention Is All You Need":**

pos — the position of the token in the sequence (0, 1, 2, ...)

i — the dimension index within the embedding vector (0 to d/2)

    -  i = slot_index // 2
       Slots 0,1 → i=0; slots 2,3 → i=1; slots 4,5 → i=2; etc.
       In other words every even, use sin. And every odd use cos

d — the total embedding dimension size

Even dimensions (2i) use sin, odd dimensions (2i+1) use cos

In [21]:
def get_angles(pos, i, embedding_dim):
    """
    Function to compute the angles for positional encoding.

    Returns the angle computed
    """
    angle_rates = pos / np.power(10000, (2 * (i//2)) / np.float32(embedding_dim))
    return angle_rates

In [22]:
def positional_encoding(position, embedding_dim):
    """
    Adds  positional encoding to the Embeddings to be fed to the Transformer model.

    Computes a sin and cos of the angles determined by the get_angles() function
    and adds the value computed to an axis of the embeddings.
    """
    positions = np.arange(position)[:, np.newaxis]          # (position, 1)
    dims = np.arange(embedding_dim)[np.newaxis, :]           # (1, embedding_dim)

    angle_rads = get_angles(positions, dims, embedding_dim)   # (position, embedding_dim)

    sines = np.sin(angle_rads[:, 0::2])
    cosines = np.cos(angle_rads[:, 1::2])

    pos_encoding = np.zeros(shape=(position, embedding_dim))
    pos_encoding[:, 0::2] = sines #evens
    pos_encoding[:, 1::2] = cosines #odds

    pos_encoding = pos_encoding[np.newaxis, ...]              # (1, position, embedding_dim) -- Use ... to indicate keep dimension shape
    return tf.cast(pos_encoding, dtype=tf.float32)

In [23]:
# Generate positional encodings
pos_encodings = positional_encoding(24, 128)
pos_encodings.shape

TensorShape([1, 24, 128])

In [24]:
# Visualize the encodings as a heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(pos_encodings[0], cmap='viridis')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position in Sequence')
plt.title('Positional Encodings')
plt.show()

/tmp/ipykernel_385/1407694799.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
#Masking

In [26]:
def create_padding_mask(seq):
    """
    Creates a padding mask for a given sequence.

    Args:
        seq (tensor): A tensor of shape (batch_size, seq_len) containing the sequence.

    Returns:
        A tensor of shape (batch_size, 1, 1, seq_len) containing a mask that is 1 where the sequence is padded, and 0 otherwise.
    """
    # Convert the sequence to a boolean tensor where True indicates a pad token (value 0).
    seq = tf.cast(tf.math.equal(seq, 0), tf.float32)

    # Add an extra dimension to the mask to add the padding to the attention logits.
    return seq[:, tf.newaxis, tf.newaxis, :]

In [27]:
def create_look_ahead_mask(size):
    """
    Creates a look-ahead mask used during training the decoder of a transformer.

    Args:
        size (int): The size of the mask.

    Returns:
        tf.Tensor: A lower triangular matrix of shape (size, size) with ones on the diagonal
            and zeros below the diagonal.
    """
    # create a matrix with ones on the diagonal and zeros below the diagonal
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0) #linalg --> linear algebra

    return mask

### Look-ahead mask

`mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)`

Used in decoder self-attention so a position can only attend to itself and earlier positions, never future ones (otherwise the model could peek at the word it is supposed to predict).

- `tf.ones((size, size))` — a size x size matrix of all 1s
- `tf.linalg.band_part(m, -1, 0)` — keeps only the lower triangle (including the diagonal), zeroing out everything above it. Row i ends up with 1s in columns 0..i, meaning "position i can see positions 0 to i"
- `1 - that` — flips it, so now row i has 1s in columns i+1 onward — these are the *future* positions row i is **not allowed** to attend to

Example for size=4:

```
0 1 1 1
0 0 1 1
0 0 0 1
0 0 0 0
```

This mask gets scaled by a large negative number and added to the attention scores before softmax, so future positions get ~0 attention weight. Usually combined with the padding mask via `tf.maximum(look_ahead_mask, padding_mask)` so decoder self-attention respects both rules at once.

In [28]:
"""

Four masks, each answering "what can this position NOT look at, and why":

enc_padding_mask (from English) — used in encoder self-attention. Blocks: English padding zeros. Reason: padding isn't a real word, don't waste attention on it.

dec_padding_mask (from English, same as #1) — used in decoder cross-attention (when decoder looks back at the encoder's output). Blocks: English padding zeros. Reason: same as #1, just applied in a different attention step.

look_ahead_mask (from French, length only) — blocks: every position after the current one. Reason: decoder must not see future words it hasn't generated yet — that's cheating during training and impossible at inference.

dec_target_padding_mask (from French) — blocks: French padding zeros. Reason: same padding concern as #1/#2, but on the target side.

Masks 3 and 4 get merged into one (combined_mask) for decoder self-attention, since that step needs both rules enforced together (no future peeking AND no padding). Masks 1 and 2 stay separate even though they're computed identically, because they're conceptually protecting two different attention operations (encoder self-attention vs. decoder cross-attention).



***** 1,2,4 are the same for positioning. 3 is just so that previous word cant see forward words *****
"""

'\n\nFour masks, each answering "what can this position NOT look at, and why":\n\nenc_padding_mask (from English) — used in encoder self-attention. Blocks: English padding zeros. Reason: padding isn\'t a real word, don\'t waste attention on it.\n\ndec_padding_mask (from English, same as #1) — used in decoder cross-attention (when decoder looks back at the encoder\'s output). Blocks: English padding zeros. Reason: same as #1, just applied in a different attention step.\n\nlook_ahead_mask (from French, length only) — blocks: every position after the current one. Reason: decoder must not see future words it hasn\'t generated yet — that\'s cheating during training and impossible at inference.\n\ndec_target_padding_mask (from French) — blocks: French padding zeros. Reason: same padding concern as #1/#2, but on the target side.\n\nMasks 3 and 4 get merged into one (combined_mask) for decoder self-attention, since that step needs both rules enforced together (no future peeking AND no padding)

In [29]:
def create_masks(inputs, targets):
    """
    Creates masks for the input sequence and target sequence.

    Args:
        inputs: Input sequence tensor.
        targets: Target sequence tensor.

    Returns:
        A tuple of three masks: the encoder padding mask, the combined mask,
        and the decoder padding mask.
    """

    # Create the encoder padding mask.
    enc_padding_mask = create_padding_mask(inputs)

    # Create the decoder padding mask.
    dec_padding_mask = create_padding_mask(inputs)

    # Create the look ahead mask for the first attention block.
    # It is used to pad and mask future tokens in the tokens received by the decoder.
    look_ahead_mask = create_look_ahead_mask(tf.shape(targets)[1])

    # Create the decoder target padding mask.
    dec_target_padding_mask = create_padding_mask(targets)

    # Combine the look ahead mask and decoder target PADDING mask for the first attention block.
    combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask) #mask outputs 1, this is for decoder-self attention

    return enc_padding_mask, combined_mask, dec_padding_mask

In [30]:
def scaled_dot_product_attention(q, k, v, mask):
    """
    Computes the scaled dot product attention weight for the query (q), key (k), and value (v) vectors.
    The attention weight is a measure of how much focus should be given to each element in the sequence of values (v)
    based on the corresponding element in the sequence of queries (q) and keys (k).

    Args:
    q: query vectors; shape (..., seq_len_q, depth)
    k: key vectors; shape  (..., seq_len_k, depth)
    v: value vectors; shape  (..., seq_len_v, depth_v)
    mask: (optional) mask to be applied to the attention weights

    Returns:
    output: The output of the scaled dot product attention computation; shape   (..., seq_len_q, depth_v)
    attention_weights: The attention weights
    """
    # Compute dot product of query and key vectors
    matmul_qk = tf.matmul(q, k, transpose_b=True) #matrix multiply (matmul), transpose so matrix can be multiplied

    # Compute the square root of the dimension of the key vectors
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_dk = tf.math.sqrt(dk) #supposed to use embedding dimension but k.shape[-1] == vector embedding length per vector

    # Compute scaled attention logits by dividing dot product by scaled dk
    scaled_attention_logits = matmul_qk / scaled_dk

    # Apply mask to the attention logits (if mask is not None)
    if mask is not None: #if there is a mask
        scaled_attention_logits += (mask * -1e9) #only applies this to make it = -10000000 when mask=1 (no token)

    # Apply softmax to the scaled attention logits to get the attention weights
    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)

    # Compute the weighted sum of the value vectors using the attention weights
    output = tf.matmul(attention_weights, v)

    return output, attention_weights



### Scaled dot-product attention

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right)V
$$

- `Q`, `K`, `V` — query, key, and value matrices, each of shape (..., seq_len, depth)
- `QK^T` — dot product between every query and every key (`tf.matmul(q, k, transpose_b=True)`), giving a raw similarity score for each query/key pair
- `d_k` — the depth (length) of each key vector; dividing by `sqrt(d_k)` keeps scores from growing too large as depth increases, which would otherwise push softmax into a near-zero-gradient regime
- `+ mask` — the padding/look-ahead mask (scaled to a large negative number) added before softmax, so blocked positions get ~0 attention weight
- `softmax(...)` — turns the scores into attention weights that sum to 1 across each row
- `... V` — the attention weights are used to compute a weighted sum of the value vectors, producing the final output

### Softmax

$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}
$$

Converts a row of raw scores into a probability distribution — all outputs are between 0 and 1, and every row sums to exactly 1.

- Larger input scores get pushed toward larger output weights, smaller/negative scores get pushed toward ~0 — the exponential exaggerates the gap between scores rather than just rescaling them linearly
- This is why the mask trick works: adding `-1e9` to a masked position makes `e^{-1e9} \approx 0`, so that position's softmax weight is effectively zero without needing special-case logic
- Applied along the last axis (over keys), so for each query position, the resulting weights tell you how much attention to pay to each other position — used directly as the weights in the weighted sum over `V`

In [31]:
class MultiHeadAttention(tf.keras.layers.Layer):
    """
    MultiHeadAttention Layer that implements the attention mechanism for the Transformer.
    It splits the input into multiple heads, computes scaled dot-product attention for each head
    and then concatenates the output of the heads and passes it through a dense layer.
    """

    def __init__(self, key_dim, num_heads, dropout_rate=0.0):
        """
        Initializes the MultiHeadAttention layer.

        Args:
            key_dim (int): The dimensionality of the key space.
            num_heads (int): The number of attention heads.
            dropout (float): The dropout rate to apply after the dense layer.
        """
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        #  ensure that the dimension of the embedding can be evenly split across attention heads
        assert key_dim % num_heads == 0
        self.depth = self.key_dim // self.num_heads

        # dense layers to project the input into queries, keys and values
        self.wq = Dense(key_dim)
        self.wk = Dense(key_dim)
        self.wv = Dense(key_dim)

        # dropout layer
        self.dropout = Dropout(dropout_rate)

        # dense layer to project the output of the attention heads
        self.dense = Dense(key_dim)

    def split_heads(self, x, batch_size):
        """
        Splits the last dimension of the tensor into (num_heads, depth).
        Transposes the result such that the shape is (batch_size, num_heads, seq_len, depth).

        Args:
            x (tensor): The tensor to be split.
            batch_size (int): The size of the batch.

        Returns:
            tensor: The tensor with the last dimension split into (num_heads, depth) and transposed.
        """
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask=None):
        """
        Applies the multi-head attention mechanism to the inputs.

        Args:
            v (tensor): The value tensor of shape (batch_size, seq_len_v, key_dim).
            k (tensor): The key tensor of shape (batch_size, seq_len_k, key_dim).
            q (tensor): The query tensor of shape (batch_size, seq_len_q, key_dim).
            mask (tensor, optional): The mask tensor of shape (batch_size, seq_len_q, seq_len_k).
                                     Defaults to None.

        Returns:
            tensor: The output tensor of shape (batch_size, seq_len_q, key_dim).
            tensor: The attention weights tensor of shape (batch_size, num_heads, seq_len_q, seq_len_k).
        """
        batch_size = tf.shape(q)[0]

        # Dense on the q, k, v vectors
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        # split the heads
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # split the queries, keys and values into multiple heads
        scaled_attention, attention_weights = scaled_dot_product_attention(q, k, v, mask)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])

        # reshape and add Dense layer
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.key_dim))
        output = self.dense(concat_attention)
        output = self.dropout(output)

        return output, attention_weights



**Why split?** One attention pass = one blended average per token.
Multiple heads let different subspaces specialize (syntax, coreference, position, etc.) — learned automatically during training, not designed by hand.

Each head sees different values of the vectors (example if 8 attention heads and 512 embedding length, each head only sees 64)

**Recombining:**
1. Concatenate heads back to `d_model` width (just placement, no mixing)
2. Final `Dense(d_model)` layer — mixes info *across* heads (this is where correlation actually happens)


In [32]:
def FeedForward(embedding_dim, fully_connected_dim):
    """Create a fully connected feedforward neural network.

    Args:
        embedding_dim (int): Dimensionality of the embedding output from the transformer layer.
        fully_connected_dim (int): Number of neurons in the fully connected layers.

    Returns:
        tf.keras.Sequential: A fully connected feedforward neural network with the specified architecture.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(fully_connected_dim, activation='relu'),
        tf.keras.layers.Dense(embedding_dim)
    ])
    return model

## Encoder

In [33]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, num_heads, fully_connected_dim, dropout_rate=0.1):
        """Initializes the encoder layer

        Args:
            embedding_dim: The dimensionality of the input and output of this layer
            num_heads: The number of attention heads to use in the multi-head attention layer
            fully_connected_dim: The dimensionality of the hidden layer in the feedforward network
            --->  The hidden/expanded size inside the feed-forward network — not the layer's input/output size
            dropout_rate: The rate of dropout to apply to the output of this layer during training

        Returns:
            A new instance of the EncoderLayer class
        """
        super(EncoderLayer, self).__init__()

        # Multi-head self-attention mechanism
        self.mha = MultiHeadAttention(embedding_dim, num_heads, dropout_rate)

        # Layer normalization
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

        # Dropout
        self.dropout = Dropout(dropout_rate)

        # Feedforward network
        self.ffn = FeedForward(embedding_dim, fully_connected_dim)

    def call(self, x, training, mask):
        """Applies the encoder layer to the input tensor

        Args:
            x: The input tensor to the encoder layer
            training: A boolean indicating whether the model is in training mode
            mask: A tensor representing the mask to apply to the attention mechanism

        Returns:
            The output of the encoder layer after applying the multi-head attention and feedforward network
        """

        # Apply multi-head self-attention mechanism to input tensor
        attn_output, _ = self.mha(x, x, x, mask=mask)

        # Apply first layer normalization and add residual connection
        out1 = self.layernorm1(attn_output + x)

        # Apply feedforward network to output of first layer normalization
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout(ffn_output, training=training)

        # Apply second layer normalization and add residual connection
        out2 = self.layernorm2(ffn_output + out1)

        return out2


$$
\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

Where, for a single vector `x` (one token's features, e.g. the 128-dim vector):

- `μ` — mean of `x`'s elements (average across the embedding dimension, computed per token, not across the batch)
- `σ²` — variance of `x`'s elements
- `ε` — tiny constant (e.g. `1e-6`) added to avoid dividing by zero if variance is ~0
- `γ`, `β` — learnable scale and shift parameters (vectors, same size as `x`), letting the network undo the normalization if that turns out to be useful for a particular feature, rather than being forced into a strict mean-0/variance-1 shape

**Why small variance doesn't cause a blow-up:**

If `σ² ≈ 0`, the values in `x` are all close to their mean, so the numerator `(x - μ)` is *also* close to 0 — they shrink together, not independently. For example, with `x = [5, 5, 5, 5]`:

- `μ = 5`, so `x - μ = [0, 0, 0, 0]`
- `σ² = 0`, so `sqrt(σ² + ε) = sqrt(0.000001) = 0.001`
- `[0, 0, 0, 0] / 0.001 = [0, 0, 0, 0]`

The result stays small because a near-zero numerator divided by a small-but-fixed denominator (never smaller than `sqrt(ε)`) is still small — `ε` only exists to prevent an exact `0/0` division-by-zero crash, not to guard against the output becoming large.

In [34]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        """
        Initializes the Encoder layer of the Transformer model.

        Args:
            num_layers (int): Number of EncoderLayers to stack.
            embedding_dim (int): Dimensionality of the token embedding space.
            num_heads (int): Number of attention heads to use in MultiHeadAttention layers.
            fully_connected_dim (int): Dimensionality of the fully connected layer in the EncoderLayer.
            input_vocab_size (int): Size of the input vocabulary.
            maximum_position_encoding (int): Maximum length of input sequences for positional encoding.
            dropout_rate (float): Probability of dropping out units during training.

        """
        super(Encoder, self).__init__()

        self.num_layers = num_layers
        self.embedding_dim = embedding_dim

        # Embedding layer
        self.embedding = Embedding(input_vocab_size, embedding_dim)

        # Positional encoding
        self.pos_encoding = positional_encoding(maximum_position_encoding, embedding_dim)

        # Encoder layers (return one encoder layer for _ in range(num_layers)
        self.enc_layers = [EncoderLayer(embedding_dim, num_heads, fully_connected_dim, dropout_rate) for _ in range(num_layers)]

        # Dropout layer
        self.dropout = Dropout(dropout_rate)

    def call(self, inputs, training, mask):
        """
        Call function for the Encoder layer.

        Args:
            inputs: tensor of shape (batch_size, sequence_length) representing input sequences
            training: boolean indicating if the model is in training mode
            mask: tensor of shape (batch_size, sequence_length) representing the mask to apply to the input sequence

        Returns:
            A tensor of shape (batch_size, sequence_length, embedding_dim) representing the encoded sequence
        """

        # Get the sequence length
        seq_len = tf.shape(inputs)[1]

        # Embed the input sequence
        inputs = self.embedding(inputs)

        # Scale the embeddings by sqrt(embedding_dim) why?
#Token IDs
#       │
#       ▼
# Embedding
#       │
#       ▼
# Multiply by √embedding_dim
#       │
#       ▼
# Add positional encoding
#       │
#       ▼
# Feed into attention layers

        inputs *= tf.math.sqrt(tf.cast(self.embedding_dim, tf.float32))

        # Add positional encodings to the input sequence
        inputs += self.pos_encoding[:, :seq_len, :]

        # Apply dropout to the input sequence
        inputs = self.dropout(inputs, training=training)

        # Pass the input sequence through each encoder layer (calls function call() from encoder_layer class), inputs becomes out2 and keep iterating each out2 into the input and keep going so on and so forth (n-block times)
        for i in range(self.num_layers):
            inputs = self.enc_layers[i](inputs, training=training, mask=mask)

        # Return the encoded sequence
        return inputs

# Classes & Constructors (`__init__`)

### Why use a class?
A class lets an object **store and remember data** (layers, parameters, configuration) so you don't have to pass everything every time.

---

### Constructor (`__init__`)
Runs **once** when the object is created.

Special function that runs automatically that sets up object's first values for its methods to work.

```python
encoder = Encoder(...)
```

Its job is to **initialize and store** the object's components.

Example:

```python
self.embedding = Embedding(...)
self.pos_encoding = positional_encoding(...)
self.enc_layers = [...]
self.dropout = Dropout(...)
```

These are stored inside the object using `self`.

---


### Class vs Function

**Class**
```python
encoder = Encoder(...)
encoder(inputs, training=True, mask=mask)
```

**Function**
```python
encoder(
    inputs,
    embedding,
    pos_encoding,
    enc_layers,
    dropout,
    training,
    mask
)
```

Functions require you to pass everything; classes remember their own data as constructor holds onto initial data (you only need to put the inputs for the method).

---

### Analogy ☕

`__init__` = Build the coffee machine once.

```text
Install grinder
Install water tank
Install filter
Install heater
```

`call()` = Brew coffee with new ingredients each time.

```python
coffee_machine.brew(coffee_beans, water)
```

You don't reinstall the grinder or heater every time—they're already part of the machine. (this would be function example)

**Build once → Use many times.**

---

### Key Idea

- `__init__()` builds and stores the model.
- `call()` reuses the stored model to process different inputs.
- `self` gives methods access to the data stored inside that object.

In [35]:
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, num_heads, fully_connected_dim, dropout_rate=0.1):
        """
        Initializes a single decoder layer of the transformer model.

        Args:
        embedding_dim: The dimension of the embedding space.
        num_heads: The number of attention heads to use.
        fully_connected_dim: The dimension of the feedforward network.
        rate: The dropout rate for regularization.
        """
        super(DecoderLayer, self).__init__()

        # Instantiate two instances of MultiHeadAttention.
        self.mha1 = MultiHeadAttention(embedding_dim, num_heads, dropout_rate) #masked French-to-French self-attention
        self.mha2 = MultiHeadAttention(embedding_dim, num_heads, dropout_rate) #French-to-English cross-attention

        # Instantiate a fully connected feedforward network.
        self.ffn = FeedForward(embedding_dim, fully_connected_dim)

        # Instantiate three layer normalization layers with epsilon=1e-6.
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)

        # Instantiate a dropout layer for regularization.
        self.dropout3 = Dropout(dropout_rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        """
        Forward pass through the decoder layer.

        Args:
        x: The input to the decoder layer, a query vector.
        enc_output: The output from the top layer of the encoder, a set of attention vectors k and v.
        training: Whether to apply dropout regularization.
        look_ahead_mask: The mask to apply to the input sequence so that it can't look ahead to future positions.
        padding_mask: The mask to apply to the input sequence to ignore padding tokens.

        Returns:
        The output from the decoder layer, a tensor with the same shape as the input.
        The attention weights from the first multi-head attention layer.
        The attention weights from the second multi-head attention layer.
        """

        # Apply the first multi-head attention layer to the query vector x.
        # We pass x as all three inputs to the layer because this is a self-attention layer.
        attn1, attn_weights_block1 = self.mha1(x, x, x, mask=look_ahead_mask)

        # Add the original input to the output of the attention layer and apply layer normalization.
        out1 = self.layernorm1(attn1 + x)

        # Apply the second multi-head attention layer to the output from the first layer and the encoder output.
        attn2, attn_weights_block2 = self.mha2(enc_output, enc_output, out1, mask=padding_mask)

        # Add the output from the first layer to the output of the second layer and apply layer normalization.
        out2 = self.layernorm2(attn2 + out1)

        # Apply the feedforward network to the output of the second layer and apply dropout regularization.
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)

        # Add the output from the second layer to the output of the feedforward network and apply layer normalization.
        out3 = self.layernorm3(ffn_output + out2)

        return out3, attn_weights_block1, attn_weights_block2


In [36]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, target_vocab_size, maximum_position_encoding, dropout_rate=0.1):
        """
        Initializes the Decoder object.

        Args:
            num_layers (int): The number of Decoder layers.
            embedding_dim (int): The size of the embedding dimension.
            num_heads (int): The number of heads in the MultiHeadAttention layer.
            fully_connected_dim (int): The number of units in the feedforward network.
            target_vocab_size (int): The number of words in the target vocabulary.
            maximum_position_encoding (int): The maximum length of a sequence.
            dropout_rate (float): The rate at which to apply dropout.
        """
        super(Decoder, self).__init__()

        self.num_layers = num_layers
        self.embedding_dim = embedding_dim

        # create layers
        self.embedding = Embedding(target_vocab_size, embedding_dim)
        self.pos_encoding = positional_encoding(maximum_position_encoding, embedding_dim)
        self.dec_layers = [DecoderLayer(embedding_dim, num_heads, fully_connected_dim, dropout_rate=0.1) for _ in range(num_layers)]
        self.dropout = Dropout(dropout_rate)

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        """
        Executes the Decoder.

        Args:
            x (tf.Tensor): The input to the Decoder.
            enc_output (tf.Tensor): The output from the Encoder.
            training (bool): Whether the Decoder is in training mode.
            look_ahead_mask (tf.Tensor): The mask for self-attention in the MultiHeadAttention layer.
            padding_mask (tf.Tensor): The mask for padding in the MultiHeadAttention layer.

        Returns:
            tf.Tensor: The output from the Decoder.
            dict: A dictionary of attention weights.
        """
        seq_len = tf.shape(x)[1]
        attention_weights = {}

        # add embedding and positional encoding
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.embedding_dim, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        x = self.dropout(x, training=training)

        # apply each layer of the decoder
        for i in range(self.num_layers):
            # pass through decoder layer i
            x, block1, block2 = self.dec_layers[i](x, enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)

            # record attention weights for block1 and block2
            attention_weights[f"decoder_layer{i + 1}_block1"] = block1
            attention_weights[f"decoder_layer{i + 1}_block2"] = block2

        return x, attention_weights

In [37]:
class Transformer(tf.keras.Model):
    """
    A Transformer model that takes in an input and target sequence and outputs a final prediction.

    Args:
        num_layers (int): Number of layers in the Encoder and Decoder.
        embedding_dim (int): Dimensionality of the embedding layer.
        num_heads (int): Number of attention heads used in the Transformer.
        fully_connected_dim (int): Dimensionality of the fully connected layer in the Encoder and Decoder.
        input_vocab_size (int): Size of the input vocabulary.
        target_vocab_size (int): Size of the target vocabulary.
        max_positional_encoding_input (int): Maximum length of the input sequence.
        max_positional_encoding_target (int): Maximum length of the target sequence.
        dropout_rate (float, optional): Dropout rate used in the Encoder and Decoder layers. Defaults to 0.1.
    """
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, target_vocab_size, max_positional_encoding_input, max_positional_encoding_target, dropout_rate=0.1):
        super(Transformer, self).__init__()

        # Initialize the Encoder and Decoder layers
        self.encoder = Encoder(num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, max_positional_encoding_input, dropout_rate)
        self.decoder = Decoder(num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, max_positional_encoding_target, dropout_rate)

        # Add a final dense layer to make the final prediction
        self.final_layer = tf.keras.layers.Dense(target_vocab_size, activation='softmax')

    def call(self, inp, tar, training, enc_padding_mask, look_ahead_mask, dec_padding_mask):
        """
        Perform a forward pass through the Transformer model.

        Args:
            inp (tf.Tensor): Input sequence tensor with shape (batch_size, input_seq_len).
            tar (tf.Tensor): Target sequence tensor with shape (batch_size, target_seq_len).
            training (bool): Whether the model is being trained or not.
            enc_padding_mask (tf.Tensor): Padding mask for the Encoder with shape (batch_size, 1, 1, input_seq_len).
            look_ahead_mask (tf.Tensor): Mask to prevent the Decoder from looking ahead in the target sequence with shape (batch_size, 1, target_seq_len, target_seq_len).
            dec_padding_mask (tf.Tensor): Padding mask for the Decoder with shape (batch_size, 1, 1, target_seq_len).

        Returns:
            tuple: A tuple containing the final output of the model and the attention weights of the Decoder.
        """
        # Pass the input sequence through the Encoder
        enc_output = self.encoder(inp, training=training, mask=enc_padding_mask)

        # Pass the target sequence and the output of the Encoder through the Decoder
        dec_output, attention_weights = self.decoder(tar, enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=dec_padding_mask)

        # Pass the output of the Decoder through the final dense layer to get the final prediction
        final_output = self.final_layer(dec_output)
        #During training, you project all decoder output vectors into vocabulary space, then softmax each one.

        return final_output, attention_weights


# Training

In [38]:
# Set hyperparameters for the Transformer model
embedding_dim = 256  # dimensionality of the embeddings used for tokens in the input and target sequences
fully_connected_dim = 512  # dimensionality of the hidden layer of the feedforward neural network within the Transformer block
num_layers = 4  # number of Transformer blocks in the encoder and decoder stacks
num_heads = 8  # number of heads in the multi-head attention mechanism
dropout_rate = 0.1  # dropout rate for regularization

# Set vocabulary sizes for input and target sequences
input_vocab_size = len(tokenizer_fr.word_index) + 2  # add 2 for the start and end tokens
target_vocab_size = len(tokenizer_en.word_index) + 2  # add 2 for the start and end tokens

# Set maximum positional encoding values for input and target sequences
max_positional_encoding_input = input_vocab_size  # maximum positional encoding value for input sequence
max_positional_encoding_target = target_vocab_size  # maximum positional encoding value for target sequence

# Set the number of epochs and batch size for training
EPOCHS = 100  # ceiling -- EarlyStopping below may stop training sooner
batch_size = 64

In a Transformer, training uses **all decoder output vectors** because each target position has a known next-token label, so the model can predict every next token in parallel. During inference, we use only the **last decoder output vector** because we are generating just the next unknown token. The final decoder vectors are projected into vocabulary space with a vocab matrix and softmax. For the learning rate schedule, the learning rate first increases linearly during warmup, reaches its maximum when `step = warmup_steps`, then decays with an inverse-square-root curve.

$$
LR(s)
=
d_{\text{model}}^{-\frac{1}{2}}
\min
\left(
s^{-\frac{1}{2}},
s \cdot w^{-\frac{3}{2}}
\right)
$$

where

$$
s = \text{step}, \qquad
w = \text{warmup\_steps}, \qquad
d_{\text{model}} = \text{model dimension}
$$

Piecewise:

$$
LR(s)
=
\begin{cases}
\frac{s}{\sqrt{d_{\text{model}}}\,w^{3/2}}, & s < w \quad \text{linear warmup} \\[8pt]
\frac{1}{\sqrt{d_{\text{model}}w}}, & s = w \quad \text{same / peak point} \\[8pt]
\frac{1}{\sqrt{d_{\text{model}}s}}, & s > w \quad \text{inverse-square-root decay}
\end{cases}
$$

In [39]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """
    A custom learning rate schedule that uses a combination of
    a square root inverse decay and a warmup schedule.

    Args:
        embedding_dim (int): The dimension of the embedding.
        warmup_steps (int): The number of steps used for warmup.

    Returns:
        float: The learning rate value at a given step.
    """
    def __init__(self, embedding_dim, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.embedding_dim = tf.cast(embedding_dim, dtype=tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, dtype=tf.float32)

    def __call__(self, step):
        """
        Compute the learning rate value for a given step using
        a combination of square root inverse decay and warmup.

        Args:
            step (int): The current step number.

        Returns:
            float: The learning rate value at the current step.
        """
        step = tf.cast(step, dtype=tf.float32)
        decay = tf.math.rsqrt(step)
        warmup = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.embedding_dim) * tf.math.minimum(decay, warmup)

# Create an instance of the custom learning rate schedule
learning_rate = CustomSchedule(embedding_dim)


## Loss Function, Optimizer, and Metrics

During training, the model uses the **Adam optimizer** to update its weights. The learning rate used by Adam is controlled by a custom Transformer learning rate schedule.

The model is trained using **Sparse Categorical Crossentropy**, which compares the predicted token distribution with the true target token at each time step.

For one prediction, cross entropy is:

$$
\mathcal{L}
=
-\sum_{i=1}^{C} y_i \log(\hat{y}_i)
$$

where:

$$
C = \text{number of classes / vocabulary size}
$$

$$
y_i = \text{true label for class } i
$$

$$
\hat{y}_i = \text{predicted probability for class } i
$$

In normal categorical crossentropy, the true label is usually written as a one-hot vector. For example, if there are 4 classes and the correct class is class 2, the label could be:

$$
y = [0,\ 0,\ 1,\ 0]
$$

The model might predict:

$$
\hat{y} = [0.05,\ 0.10,\ 0.80,\ 0.05]
$$

The cross entropy becomes:

$$
\mathcal{L}
=
-\left(
0\log(0.05)
+
0\log(0.10)
+
1\log(0.80)
+
0\log(0.05)
\right)
$$

All terms multiplied by 0 disappear, so:

$$
\mathcal{L}
=
-\log(0.80)
$$

Sparse categorical crossentropy uses an integer label instead of a one-hot vector. So instead of writing:

$$
y = [0,\ 0,\ 1,\ 0]
$$

we write only the correct class index:

$$
k = 2
$$

Then the loss is:

$$
\mathcal{L}
=
-\log(\hat{y}_k)
$$

where:

$$
\hat{y}_k = \text{predicted probability of the correct class}
$$

So sparse categorical crossentropy is mathematically the same idea as categorical crossentropy, but it stores the true label more efficiently as an integer class index.

If the model gives high probability to the correct class, the loss is small:

$$
-\log(0.80) \approx 0.223
$$

If the model gives low probability to the correct class, the loss is large:

$$
-\log(0.01) \approx 4.605
$$

For a target sequence, the loss is computed at every time step and averaged:

$$
\mathcal{L}_{seq}
=
\frac{1}{T}
\sum_{t=1}^{T}
-\log P(y_t \mid y_{<t}, x)
$$

where:

$$
T = \text{target sequence length / the number of token positions in the decoder target sentence}
$$

$$
y_t = \text{true token at time step } t
$$

$$
P(y_t \mid y_{<t}, x) = \text{model probability assigned to the correct token}
$$

$$
\text{Summary: Compute loss for each target token position,
add them together,
divide by the number of target token positions.}
$$

After the loss is calculated, TensorFlow uses backpropagation to compute gradients:

$$
\nabla_\theta \mathcal{L}
$$

where:

$$
\theta = \text{all trainable model parameters}
$$

Then Adam updates the model parameters using the gradients and learning rate:

$$
\theta_{\text{new}}
=
\theta_{\text{old}}
-
\eta \cdot \text{AdamUpdate}(\nabla_\theta \mathcal{L})
$$

where:

$$
\eta = \text{learning rate}
$$

The training accuracy is measured using **Sparse Categorical Accuracy**. It compares the predicted token with the true token at each time step.

The predicted class/token is:

$$
\hat{y}_t
=
\arg\max_i P(i \mid y_{<t}, x)
$$

Accuracy is:

$$
\text{Accuracy}
=
\frac{\text{number of correct predictions}}
{\text{total number of predictions}}
$$

or:

$$
\text{Accuracy}
=
\frac{1}{T}
\sum_{t=1}^{T}
\mathbf{1}(\hat{y}_t = y_t)
$$

During training, `tf.keras.metrics.Mean` tracks the average loss, and `tf.keras.metrics.SparseCategoricalAccuracy` tracks the token-level prediction accuracy.

In [40]:
# Create an instance of the Transformer model
transformer = Transformer(num_layers, embedding_dim, num_heads,
                           fully_connected_dim, input_vocab_size, target_vocab_size,
                           max_positional_encoding_input, max_positional_encoding_target, dropout_rate)

# Define the optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

# Define the loss object
loss_object = tf.keras.losses.SparseCategoricalCrossentropy()


def loss_function(true_values, predictions):
    """
    Calculate the loss value for a given target sequence.

    Args:
        true_values (tf.Tensor): The true target sequence.
        predictions (tf.Tensor): The predicted target sequence.

    Returns:
        float: The loss value for the given target sequence.
    """
    # Create a mask to exclude the padding tokens
    mask = tf.math.logical_not(tf.math.equal(true_values, 0))

    # Compute the loss value using the loss object
    loss_ = loss_object(true_values, predictions)

    # Apply the mask to exclude the padding tokens
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ = loss_ * mask

    # Calculate the mean loss value
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

def accuracy_function(true_values, predictions):
    """
    Calculate the accuracy for a given target sequence.

    Args:
        true_values (tf.Tensor): The true target sequence.
        predictions (tf.Tensor): The predicted target sequence.

    Returns:
        float: The accuracy value for the given target sequence.
    """
    # Compute the accuracies using the true and predicted target sequences
    accuracies = tf.equal(true_values, tf.argmax(predictions, axis=-1))

    # Create a mask to exclude the padding tokens
    mask = tf.math.logical_not(tf.math.equal(true_values, 0))

    # Apply the mask to exclude the padding tokens from the accuracies
    accuracies = tf.math.logical_and(mask, accuracies)
    accuracies = tf.cast(accuracies, dtype=tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)

    # Calculate the mean accuracy value
    return tf.reduce_sum(accuracies) / tf.reduce_sum(mask)

# Define the training metrics
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

# Training (Transformer II — 85/10/5 split, EarlyStopping on val_accuracy)

**Using the transformer model, the input and masks are passed through a forward pass to generate predictions. These predictions are then compared to the expected output to calculate the loss. The gradients of the loss with respect to the trainable variables of the transformer model are computed using tf.GradientTape(), and the optimizer is applied to update the model parameters.**

**Finally, the training loss and accuracy metrics are updated using the train_loss and train_accuracy functions**

In [41]:
train_step_signature = [
    tf.TensorSpec(shape=(batch_size, 24), dtype=tf.int64), #expects input
    tf.TensorSpec(shape=(batch_size,24), dtype=tf.int64), #expects target
]

#@tf.function converts the Python function into a TensorFlow computation graph. This generally makes repeated training steps faster. However, this is optional
@tf.function(input_signature=train_step_signature)
def train_step(encoder_input, target):
    """
    Function to perform a single training step.

    Args:
    encoder_input (tf.Tensor): The input tensor for the encoder.
    target (tf.Tensor): The target tensor for the decoder.

    Returns:
    None.
    """

    # Slice the target tensor to get the input for the decoder
    decoder_input = target[:, :-1]

    # Slice the target tensor to get the expected output of the decoder
    expected_output = target[:, 1:]

    # Create masks for the encoder input, decoder input and the padding
    enc_padding_mask, combined_mask, dec_padding_mask = create_masks(encoder_input, decoder_input)

    # Perform a forward pass through the model
    with tf.GradientTape() as tape: #backprop pretty much
        #TensorFlow’s mechanism for automatic differentiation—calculating gradients needed to train a model. tape is simply a variable holding the tf.GradientTape object:
        predictions, _ = transformer(encoder_input, decoder_input, training=True, enc_padding_mask=enc_padding_mask,look_ahead_mask=combined_mask, dec_padding_mask=dec_padding_mask)

        # Calculate the loss between the predicted output and the expected output
        loss = loss_function(expected_output, predictions)

    # Calculate gradients and update the model parameters
    gradients = tape.gradient(loss, transformer.trainable_variables) #property automatically provided by TensorFlow
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables)) #zip pairs corresponding items from multiple lists into one (list of gradients of how much loss will change and weights)


    #updating training loss and accuracy
    train_loss(loss)
    train_accuracy(expected_output, predictions)

1. Take one batch of source and target sequences
2. Shift target into decoder input and expected output
3. Create masks
4. Run Transformer forward pass
5. Compute cross-entropy loss
6. Backpropagate gradients
7. Update weights once
8. Update loss/accuracy metrics

```text
x = tf.constant(3.0)

with tf.GradientTape() as tape:
    tape.watch(x)
    y = x ** 2

dy_dx = tape.gradient(y, x)  # 6.0
```

## Backpropagation and Weight Updates

After the forward pass calculates the predictions and loss, TensorFlow performs backpropagation:

```python
gradients = tape.gradient(
    loss,
    transformer.trainable_variables
)
```

`GradientTape` recorded the operations used to produce the loss. The `tape.gradient()` call follows those operations backward and calculates the gradient of the loss with respect to every trainable variable:

$$
\frac{\partial \text{loss}}{\partial \text{weight}}
$$

A gradient tells us how the loss would change if its corresponding weight changed slightly.

The returned gradients and trainable variables have matching positions:

```text
gradients[0] belongs to transformer.trainable_variables[0]
gradients[1] belongs to transformer.trainable_variables[1]
gradients[2] belongs to transformer.trainable_variables[2]
...
```

The gradients are then paired with their corresponding variables:

```python
gradient_variable_pairs = zip(
    gradients,
    transformer.trainable_variables
)
```

The optimizer uses these pairs to update the weights:

```python
optimizer.apply_gradients(gradient_variable_pairs)
```

The gradient describes the slope of the loss, but it is not necessarily the exact weight update. Adam uses the gradient, learning rate, and moving averages of previous gradients to determine the actual update.

## Complete Training-Step Flow

```text
encoder input + target
          ↓
shift the target to create:
decoder input + expected output
          ↓
create attention masks
          ↓
Transformer forward pass
          ↓
generate token predictions
          ↓
compare predictions with expected output
          ↓
calculate loss
          ↓
tape.gradient() performs backpropagation
          ↓
calculate one gradient for each trainable variable
          ↓
zip() pairs each gradient with its variable
          ↓
Adam updates the trainable variables
          ↓
record training loss and accuracy
```

In short:

```text
Forward pass:  inputs → predictions → loss
Backward pass: loss → gradients for every weight
Update step:   Adam uses gradients → changes weights
Metrics:       record loss and accuracy
```

---

## Training Loop <a name="5-5"></a>

**This is the training loop where the model is trained on the input and target batches in iterations.**

**The loop starts by resetting the training metrics at the start of each epoch. Then, it iterates through the dataset in batches of size batch_size. At each iteration, it gets the input and target batch and calls the train_step function to train the model on the current batch.**

**The training metrics (loss and accuracy) are updated at each iteration using the train_loss and train_accuracy objects. These metrics are printed every 100 batches to monitor the training progress.**

**Finally, after iterating through the dataset, the epoch loss and accuracy are printed.**

## Validation / Test Step

Same forward pass and loss/accuracy computation as `train_step`, but with `training=False` and no `GradientTape`/gradient update — used to evaluate the held-out validation and test splits without changing the model's weights.

In [42]:
val_loss = tf.keras.metrics.Mean(name='val_loss')
val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')

eval_step_signature = [
    tf.TensorSpec(shape=(batch_size, 24), dtype=tf.int64),
    tf.TensorSpec(shape=(batch_size, 24), dtype=tf.int64),
]

@tf.function
def eval_step(encoder_input, target, loss_metric, accuracy_metric):
    """
    Function to evaluate the model on a batch without updating its weights.

    Args:
    encoder_input (tf.Tensor): The input tensor for the encoder.
    target (tf.Tensor): The target tensor for the decoder.
    loss_metric: The tf.keras.metrics.Mean instance to update.
    accuracy_metric: The tf.keras.metrics.SparseCategoricalAccuracy instance to update.

    Returns:
    None.
    """

    decoder_input = target[:, :-1]
    expected_output = target[:, 1:]

    enc_padding_mask, combined_mask, dec_padding_mask = create_masks(encoder_input, decoder_input)

    predictions, _ = transformer(encoder_input, decoder_input, training=False, enc_padding_mask=enc_padding_mask, look_ahead_mask=combined_mask, dec_padding_mask=dec_padding_mask)

    loss = loss_function(expected_output, predictions)

    loss_metric(loss)
    accuracy_metric(expected_output, predictions)


## Training Loop — train on the 85% split, validate on the 10% split each epoch

Training runs for up to `EPOCHS` (20) epochs, but `EarlyStopping` (set up below) may stop it sooner once validation accuracy stops improving.

## Early Stopping on Validation Accuracy

Instead of manually reading a train-vs-validation accuracy graph and re-running training up to a hand-picked epoch, use Keras's built-in `EarlyStopping` callback: `monitor='val_accuracy'`, `patience=5`, `mode='max'`, `restore_best_weights=True`. Training stops once validation accuracy hasn't improved for 5 consecutive epochs, and the model's weights are automatically rolled back to whichever epoch had the single best validation accuracy — no manual re-run needed.

Since this notebook uses a manual `tf.GradientTape` training loop rather than `model.fit()`, the callback has to be driven by hand: `set_model()` + `on_train_begin()` before the loop, `on_epoch_end()` after each epoch (checking `transformer.stop_training` to know when to break), and `on_train_end()` after the loop — that last call is what actually restores the best weights.

In [43]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    mode='max',
    restore_best_weights=True,
    verbose=1,
)
early_stopping.set_model(transformer)
transformer.stop_training = False
early_stopping.on_train_begin()


In [44]:
train_loss_history = []
train_accuracy_history = []
val_loss_history = []
val_accuracy_history = []

for epoch in range(1, EPOCHS + 1):
    # Reset the metrics at the start of the next epoch
    train_loss.reset_state()
    train_accuracy.reset_state()
    val_loss.reset_state()
    val_accuracy.reset_state()

    # --- Train on the 85% training split ---
    current_batch_index = 0
    for i in range(len(english_train) // batch_size):
        target_batch = tf.convert_to_tensor(np.array(english_train[current_batch_index:current_batch_index + batch_size]), dtype=tf.int64)
        input_batch = tf.convert_to_tensor(np.array(french_train[current_batch_index:current_batch_index + batch_size]), dtype=tf.int64)

        current_batch_index += batch_size
        train_step(input_batch, target_batch)

    # --- Validate on the 10% validation split (no weight updates) ---
    current_batch_index = 0
    for i in range(len(english_val) // batch_size):
        target_batch = tf.convert_to_tensor(np.array(english_val[current_batch_index:current_batch_index + batch_size]), dtype=tf.int64)
        input_batch = tf.convert_to_tensor(np.array(french_val[current_batch_index:current_batch_index + batch_size]), dtype=tf.int64)

        current_batch_index += batch_size
        eval_step(input_batch, target_batch, val_loss, val_accuracy)

    print(f'Epoch {epoch} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f} '
          f'Val Loss {val_loss.result():.4f} Val Accuracy {val_accuracy.result():.4f}')

    train_loss_history.append(float(train_loss.result()))
    train_accuracy_history.append(float(train_accuracy.result()))
    val_loss_history.append(float(val_loss.result()))
    val_accuracy_history.append(float(val_accuracy.result()))

    # Let EarlyStopping see this epoch's validation metrics and decide whether to stop.
    early_stopping.on_epoch_end(epoch, logs={
        'val_accuracy': val_accuracy_history[-1],
        'val_loss': val_loss_history[-1],
    })
    if transformer.stop_training:
        print(f'Early stopping triggered after epoch {epoch} (best val_accuracy epoch will be restored).')
        break

# Required for restore_best_weights=True to actually copy the best epoch's weights back into the model.
early_stopping.on_train_end()
print(f'EarlyStopping.stopped_epoch = {early_stopping.stopped_epoch} (0 means it never triggered)')


/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'multi_head_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'encoder_layer' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'multi_head_attention_1' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'encoder_layer_1' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'multi_head_attention_2' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the 

/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'encoder_layer_2' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'multi_head_attention_3' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'encoder_layer_3' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask informat

/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'encoder' (of type Encoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'multi_head_attention_4' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/rkim8/translation-venv/lib/python3.12/site-packages/keras/src/layers/layer.py:1039: UserWarning: Layer 'multi_head_attention_5' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


...

Epoch 28 Loss 0.0171 Accuracy 0.9889 Val Loss 0.0167 Val Accuracy 0.9888


Epoch 29 Loss 0.0170 Accuracy 0.9889 Val Loss 0.0166 Val Accuracy 0.9888


Epoch 30 Loss 0.0173 Accuracy 0.9889 Val Loss 0.0167 Val Accuracy 0.9890
Early stopping triggered after epoch 30 (best val_accuracy epoch will be restored).
Epoch 31: early stopping


Restoring model weights from the end of the best epoch: 26.


EarlyStopping.stopped_epoch = 30 (0 means it never triggered)


## Train vs Validation Accuracy per Epoch

Visualizes the same bias/variance tradeoff `EarlyStopping` used to decide when to stop — the dashed line marks the epoch with the best validation accuracy, which is the epoch whose weights were restored into the model.

In [45]:
best_epoch = int(np.argmax(val_accuracy_history)) + 1

plt.figure(figsize=(10, 6))
epochs_range = range(1, len(train_accuracy_history) + 1)
plt.plot(epochs_range, train_accuracy_history, marker="o", label="Train Accuracy")
plt.plot(epochs_range, val_accuracy_history, marker="o", label="Validation Accuracy")
plt.axvline(best_epoch, color="gray", linestyle="--", label=f"Best epoch ({best_epoch}) — restored weights")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Validation Accuracy per Epoch")
plt.legend()
plt.show()


/tmp/ipykernel_385/3923154729.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final Test-Set Evaluation — 5% held-out split, evaluated once after training

In [46]:
test_loss.reset_state()
test_accuracy.reset_state()

current_batch_index = 0
for i in range(len(english_test) // batch_size):
    target_batch = tf.convert_to_tensor(np.array(english_test[current_batch_index:current_batch_index + batch_size]), dtype=tf.int64)
    input_batch = tf.convert_to_tensor(np.array(french_test[current_batch_index:current_batch_index + batch_size]), dtype=tf.int64)

    current_batch_index += batch_size
    eval_step(input_batch, target_batch, test_loss, test_accuracy)

print(f'Test Loss {test_loss.result():.4f} Test Accuracy {test_accuracy.result():.4f}')


Test Loss 0.0170 Test Accuracy 0.9887


In [47]:
transformer.save_weights("transformer3_weights.weights.h5") #save_weights() and load_weights() are built-in Keras model features.


In [48]:
import json

history = {
    "train_loss": train_loss_history,
    "train_accuracy": train_accuracy_history,
    "val_loss": val_loss_history,
    "val_accuracy": val_accuracy_history,
    "test_loss": float(test_loss.result()),
    "test_accuracy": float(test_accuracy.result()),
    "stopped_epoch": early_stopping.stopped_epoch,
}

with open("transformer3_history.json", "w") as f:
    json.dump(history, f, indent=2)
